In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split
from utils import load_galaxy_data
import app


In [ ]:
input_data, labels = load_galaxy_data()

input_data, labels = load_galaxy_data()

print("Input data shape:", input_data.shape)
print("Labels shape:", labels.shape)


In [ ]:
#Train/Validation split
X_train, X_val, y_train, y_val = train_test_split(
    input_data,
    labels,
    test_size=0.20,
    shuffle=True,
    stratify=labels,
    random_state=222
)

#Preprocessing
datagen = ImageDataGenerator(rescale=1./255)

#Iterators
train_iterator = datagen.flow(X_train, y_train, batch_size=5, shuffle=True)
val_iterator = datagen.flow(X_val, y_val, batch_size=5, shuffle=False)


In [ ]:
model = Sequential([
    layers.Conv2D(8, (3,3), strides=2, activation='relu', input_shape=X_train.shape[1:]),
    layers.MaxPooling2D(pool_size=(2,2), strides=2),
    layers.Conv2D(8, (3,3), strides=2, activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=2),
    layers.Flatten(),
    layers.Dense(16, activation='relu'),
    layers.Dense(4, activation='softmax')  # 4 classes
])
#Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[tf.keras.metrics.CategoricalAccuracy(), tf.keras.metrics.AUC()]
)


In [ ]:
#Summary
model.summary()
steps_per_epoch = len(X_train) // 5
val_steps = len(X_val) // 5

history = model.fit(
    train_iterator,
    steps_per_epoch=steps_per_epoch,
    epochs=8,
    validation_data=val_iterator,
    validation_steps=val_steps
)


In [ ]:
#Evaluate
results = model.evaluate(val_iterator, steps=val_steps)
print("Validation Results:", results)

from visualize import visualize_activations
visualize_activations(model, val_iterator)